In [ ]:
import torch
import pickle as pkl
from descope.inference import InferenceForRNA

### 1. Initialize Inference Tool

In [ ]:
celltype = "A549"

infer = InferenceForRNA(
    test_csv_template_fp=f"./test_info_{celltype}.csv",
    adata=f"./{celltype}_ground_truth.h5ad",
    pretrained_model_name_or_path="./descope-tahoe100m-12059-hvgs",
    pert_col="drug_and_dose",
    ctrl_name="control_0.0",
    target_sum=1e4,
    skip_raw_counts_check=False  # Avoid doing log-normalization twice
)

### 2. Start Inference Process

- **Note:**
    
    - **!!! Always use real control cells, i.e., set `use_generated_control_cells=False` !!!**

    - **!!! Never set `use_generate_control_cells=True` !!!**

In [ ]:
infer.inference(
    gene_embs_file="./drug_dose_embed.pt",
    device=torch.device("cuda:0"),
    batch_size=256,
    use_generated_control_cells=False
)

### 3. Calculate Metrics

#### 3.1 Compute the built-in metrics of cell-eval

In [ ]:
# Select metrics you want to calculate
infer.all_metrics

In [ ]:
metrics_to_calculate = [
    "pearson_delta",
    "mse", "mae", "mse_delta", "mae_delta",
    "discrimination_score_l1",
    "overlap_at_N", 
    "overlap_at_50", 
    "overlap_at_100", 
    "overlap_at_200", 
    "overlap_at_500",
    "precision_at_N", 
    "precision_at_50", 
    "precision_at_100", 
    "precision_at_200", 
    "precision_at_500",
    "de_direction_match", 
    "de_spearman_lfc_sig",
    "pr_auc", "roc_auc"
]

# Performance metrics are evaluated strictly on the subset of genes reliably predicted by the model.
# Since the model cannot predict genes outside the effective_genes set, 
# you can directly copy the expression values from the training set for these genes when evaluating against other methods.
# For simplicity, we restrict our evaluation to effective_genes only.
with open("./reliably_genes.pkl", "rb") as f:
    effective_genes = pkl.load(f)
    
infer.adata_pred = infer.adata_pred[:, effective_genes]
infer.adata_test_ground_truth = infer.adata_test_ground_truth[:, effective_genes]

results, agg_results, evaluator = infer.compute_metrics(
    adata_pred=infer.adata_pred,
    adata_real=infer.adata_test_ground_truth,
    metrics_to_calculate=metrics_to_calculate,
    de_pred=None,
    de_real=None,
    control_pert=infer.ctrl_name,
    pert_col=infer.pert_col,
    de_method="wilcoxon",
    num_threads=32,
    outdir=f"./cell-eval-outdir/{celltype}"
)

In [ ]:
agg_results

#### 3.2 Compute Extra Metrics

In [ ]:
# pearson
pearson, pearson_mean = infer.extra_metrics_func.pearson(evaluator.anndata_pair)
pearson_mean

In [ ]:
# person_delta_on_topk_de
pearson_delta_on_topk_de, pearson_delta_on_topk_de_mean = infer.extra_metrics_func.pearson_delta_on_topk_de(
    data=evaluator.anndata_pair,
    de_real=f"./cell-eval-outdir/{celltype}/real_de.csv",
    topk=20
)
pearson_delta_on_topk_de_mean

In [ ]:
# direction_match_on_topk_de
up, down, up_mean, down_mean = infer.extra_metrics_func.direction_match_on_topk_de(
    data=evaluator.anndata_pair,
    de_real=f"./cell-eval-outdir/{celltype}/real_de.csv",
    topk=100,
    separate_up_down_regulated=True
)
up_mean, down_mean

### 4. Aggregate Results

In [ ]:
results = results.to_pandas()
results["pearson"] = results["perturbation"].map(pearson)
results["pearson_delta_on_topk_de"] = results["perturbation"].map(pearson_delta_on_topk_de)
results["direction_match_on_topk_up_de"] = results["perturbation"].map(up)
results["direction_match_on_topk_down_de"] = results["perturbation"].map(down)
results.to_csv(f"./cell-eval-outdir/{celltype}/results_for_each_perturbation.csv")

### 5. Save Predicted AnnData

In [ ]:
infer.write_h5ad(
    adata=infer.adata_pred,
    save_path=f"./cell-eval-outdir/{celltype}/descope_preds.h5ad"
)